In [2]:
import requests
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import os

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.linear_model import LinearRegression, RidgeCV, LassoCV

from reportlab.lib.pagesizes import letter
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle, PageBreak
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib import colors
from reportlab.lib.units import inch


In [3]:
# Import Savant data
headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/58.0.3029.110 Safari/537.3'}
savant = "https://baseballsavant.mlb.com/team/144"
response = requests.get(savant, headers=headers)
sav_tables = pd.read_html(response.text)

standard = []
statcast = []
yahoo = []
sprint = []

standard = pd.DataFrame(sav_tables[0])
standard.columns = standard.columns.droplevel(0)
statcast = pd.merge(sav_tables[1],sav_tables[2])
sprint = pd.DataFrame(sav_tables[6])
statcast = pd.merge(statcast, standard)

# Import Yahoo data
yahoo = "https://sports.yahoo.com/mlb/teams/atlanta/stats/"
yah_tables = pd.read_html(yahoo)
yahoo = pd.DataFrame(yah_tables[0])

C:\Users\nncg7\AppData\Local\Temp\ipykernel_12052\2366039330.py:5: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  sav_tables = pd.read_html(response.text)


In [4]:
# Cleaning Data + Making MB Value
def clean_player_names(df, flip_names=True):
    df['Player'] = df['Player'].str.replace(',', '').str.replace('Jr.', '').str.replace('II ', '').str.strip()
    df.columns = df.columns.str.replace(' ', '_')
    if flip_names:
        df['Player'] = df['Player'].str.split(' ', n=1).str[::-1].str.join(' ')
    return df

statcast = clean_player_names(statcast)
sprint = clean_player_names(sprint, flip_names=False)
yahoo = clean_player_names(yahoo, flip_names=False)

if standard.iloc[-1]['Player'] == 'MLB':
    standard = standard[:-2]
    standard.sort_values('Player', ascending=False)
if statcast.iloc[-1]['Player'] == 'MLB':
    statcast = statcast[:-2]
    standard.sort_values('Player', ascending=False)

sprint = sprint.sort_values('Player', ascending=False)
yahoo = yahoo.sort_values('Player', ascending=False)
yahoo = pd.merge(yahoo, sprint)

yahoo['MB_Value'] = round(((yahoo['H'] + yahoo['BB'] - yahoo['CS']) * 
                     (((yahoo['H'] - yahoo['2B'] - yahoo['3B'] - yahoo['HR']) + 2 * yahoo['2B'] + 3 * yahoo['3B'] + 4 * yahoo['HR']) 
                      + 0.7 * yahoo['SB'])) / (yahoo['AB'] + yahoo['BB'] + yahoo['CS']),3)
statcast['TB'] = ((statcast['H'] - statcast['2B'] - statcast['3B'] - statcast['HR']) + 2 * statcast['2B'] + 3 * statcast['3B'] + 4 * statcast['HR'])

In [16]:
# Create CSV File
statcast.to_csv('Single_Statcast.csv', index=False)
yahoo.to_csv('Single_Yahoo.csv', index=False)
os.startfile('Single_Yahoo.csv')
os.startfile("Single_Statcast.csv")

In [6]:
# Research SB
corr_SB = yahoo[['SB','CS','Sprint_Speed_(ft/s)','Lg_Rank', 'Pos_Rank', 'Age_Rank', '%_Rank']].corr()['SB']
selected_features = corr_SB[(corr_SB.abs() > 0.4) & (corr_SB.index != 'SB')].index.tolist()
print("Selected features:", selected_features)

X = yahoo[selected_features].dropna()
y = yahoo.loc[X.index, 'AB']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = LinearRegression()
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
print(f"R² Score:  {r2_score(y_test, y_pred):.4f}")
print(f"RMSE:      {np.sqrt(mean_squared_error(y_test, y_pred)):.4f}")

coef_df = pd.DataFrame({
    'Feature': selected_features,
    'Coefficient': model.coef_
}).sort_values('Coefficient', key=abs, ascending=False)
print(coef_df)
print(f"Intercept: {model.intercept_:.4f}")

Selected features: ['CS', 'Sprint_Speed_(ft/s)', 'Lg_Rank', '%_Rank']
R² Score:  -3.0359
RMSE:      230.3155
               Feature  Coefficient
1  Sprint_Speed_(ft/s)  -214.903990
3               %_Rank   121.409391
0                   CS    30.534565
2              Lg_Rank    18.553924
Intercept: -5620.5525


In [7]:
# Research CS
corr_CS = yahoo[['SB','CS','Sprint_Speed_(ft/s)','Lg_Rank', 'Pos_Rank', 'Age_Rank', '%_Rank']].corr()['CS']
selected_features = corr_CS[(corr_CS.abs() > 0.4) & (corr_CS.index != 'CS')].index.tolist()
print("Selected features:", selected_features)

X = yahoo[selected_features].dropna()
y = yahoo.loc[X.index, 'CS']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = LinearRegression()
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
print(f"R² Score:  {r2_score(y_test, y_pred):.4f}")
print(f"RMSE:      {np.sqrt(mean_squared_error(y_test, y_pred)):.4f}")

coef_df = pd.DataFrame({
    'Feature': selected_features,
    'Coefficient': model.coef_
}).sort_values('Coefficient', key=abs, ascending=False)
print(coef_df)
print(f"Intercept: {model.intercept_:.4f}")

Selected features: ['SB']
R² Score:  0.0000
RMSE:      0.4418
  Feature  Coefficient
0      SB      0.24245
Intercept: 0.4418


In [8]:
# Research AB
corr_AB = statcast[['Pitches', 'Zone_%', 'Zone_Swing_%',
       'Zone_Contact_%', 'Chase_%', 'Chase_Contact_%', 'Edge_%',
       '1st_Pitch_Swing_%', 'Swing_%', 'Whiff_%', 'Meatball_%',
       'Meatball_Swing_%', 'GB_%', 'FB_%', 'LD_%', 'PU_%', 'Pull_%',
       'Straight_%', 'Oppo_%', 'Weak_%', 'Topped_%', 'Under_%',
       'Flare/Burner_%', 'Solid_%', 'Launch_Angle_Sweet-Spot_%', 'Barrel_%',
       'Batted_Balls', 'Barrels', 'Hard_Hit_%',
       'Exit_Velocity', 'Launch_Angle', 'AB']].corr()['AB']
selected_features = corr_AB[(corr_AB.abs() > 0.5) & (corr_AB.index != 'AB')].index.tolist()
print("Selected features:", selected_features)

X = statcast[selected_features].dropna()
y = statcast.loc[X.index, 'AB']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = LinearRegression()
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
print(f"R² Score:  {r2_score(y_test, y_pred):.4f}")
print(f"RMSE:      {np.sqrt(mean_squared_error(y_test, y_pred)):.4f}")

coef_df = pd.DataFrame({
    'Feature': selected_features,
    'Coefficient': model.coef_
}).sort_values('Coefficient', key=abs, ascending=False)
print(coef_df)
print(f"Intercept: {model.intercept_:.4f}")


Selected features: ['Pitches', 'Zone_%', 'Chase_%', 'Barrel_%', 'Batted_Balls', 'Barrels', 'Hard_Hit_%']
R² Score:  0.0333
RMSE:      60.6791
        Feature  Coefficient
3      Barrel_%     4.932893
6    Hard_Hit_%    -4.014964
2       Chase_%     3.255893
1        Zone_%     2.044240
5       Barrels     1.415539
4  Batted_Balls     0.613934
0       Pitches     0.085703
Intercept: -55.0984


In [9]:
# Research H
corr_H = statcast[['Pitches', 'Zone_%', 'Zone_Swing_%',
       'Zone_Contact_%', 'Chase_%', 'Chase_Contact_%', 'Edge_%',
       '1st_Pitch_Swing_%', 'Swing_%', 'Whiff_%', 'Meatball_%',
       'Meatball_Swing_%', 'GB_%', 'FB_%', 'LD_%', 'PU_%', 'Pull_%',
       'Straight_%', 'Oppo_%', 'Weak_%', 'Topped_%', 'Under_%',
       'Flare/Burner_%', 'Solid_%', 'Launch_Angle_Sweet-Spot_%', 'Barrel_%',
       'Batted_Balls', 'Barrels', 'Hard_Hit_%',
       'Exit_Velocity', 'Launch_Angle','H']].corr()['H']
selected_features = corr_H[(corr_H.abs() > 0.5) & (corr_H.index != 'H')].index.tolist()
print("Selected features:", selected_features)

X = statcast[selected_features].dropna()
y = statcast.loc[X.index, 'H']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = LinearRegression()
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
print(f"R² Score:  {r2_score(y_test, y_pred):.4f}")
print(f"RMSE:      {np.sqrt(mean_squared_error(y_test, y_pred)):.4f}")

coef_df = pd.DataFrame({
    'Feature': selected_features,
    'Coefficient': model.coef_
}).sort_values('Coefficient', key=abs, ascending=False)
print(coef_df)
print(f"Intercept: {model.intercept_:.4f}")

Selected features: ['Pitches', 'Zone_%', 'Barrel_%', 'Batted_Balls', 'Barrels', 'Hard_Hit_%']
R² Score:  0.6362
RMSE:      8.9874
        Feature  Coefficient
2      Barrel_%    -1.462511
4       Barrels     0.837317
1        Zone_%     0.624090
5    Hard_Hit_%     0.400794
3  Batted_Balls     0.202341
0       Pitches     0.007666
Intercept: -34.5902


In [10]:
# Research BB
corr_BB = statcast[['Pitches', 'Zone_%', 'Zone_Swing_%',
       'Zone_Contact_%', 'Chase_%', 'Chase_Contact_%', 'Edge_%',
       '1st_Pitch_Swing_%', 'Swing_%', 'Whiff_%', 'Meatball_%',
       'Meatball_Swing_%', 'GB_%', 'FB_%', 'LD_%', 'PU_%', 'Pull_%',
       'Straight_%', 'Oppo_%', 'Weak_%', 'Topped_%', 'Under_%',
       'Flare/Burner_%', 'Solid_%', 'Launch_Angle_Sweet-Spot_%', 'Barrel_%',
       'Batted_Balls', 'Barrels', 'Hard_Hit_%',
       'Exit_Velocity', 'Launch_Angle', 'BB']].corr()['BB']
selected_features = corr_BB[(corr_BB.abs() > 0.5) & (corr_BB.index != 'BB')].index.tolist()
print("Selected features:", selected_features)

X = statcast[selected_features].dropna()
y = statcast.loc[X.index, 'BB']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = LinearRegression()
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
print(f"R² Score:  {r2_score(y_test, y_pred):.4f}")
print(f"RMSE:      {np.sqrt(mean_squared_error(y_test, y_pred)):.4f}")

coef_df = pd.DataFrame({
    'Feature': selected_features,
    'Coefficient': model.coef_
}).sort_values('Coefficient', key=abs, ascending=False)
print(coef_df)
print(f"Intercept: {model.intercept_:.4f}")

Selected features: ['Pitches', 'Zone_%', 'Zone_Contact_%', 'Barrel_%', 'Batted_Balls', 'Barrels', 'Hard_Hit_%']
R² Score:  -406.8272
RMSE:      115.4593
          Feature  Coefficient
6      Hard_Hit_%     3.518408
1          Zone_%    -3.490065
3        Barrel_%    -2.640697
5         Barrels    -2.199019
2  Zone_Contact_%    -1.934252
4    Batted_Balls    -0.121204
0         Pitches     0.074295
Intercept: 225.8987


In [11]:
# Research TB
corr_TB = statcast[['Pitches', 'Zone_%', 'Zone_Swing_%',
       'Zone_Contact_%', 'Chase_%', 'Chase_Contact_%', 'Edge_%',
       '1st_Pitch_Swing_%', 'Swing_%', 'Whiff_%', 'Meatball_%',
       'Meatball_Swing_%', 'GB_%', 'FB_%', 'LD_%', 'PU_%', 'Pull_%',
       'Straight_%', 'Oppo_%', 'Weak_%', 'Topped_%', 'Under_%',
       'Flare/Burner_%', 'Solid_%', 'Launch_Angle_Sweet-Spot_%', 'Barrel_%',
       'Batted_Balls', 'Barrels', 'Hard_Hit_%',
       'Exit_Velocity', 'Launch_Angle', 'TB']].corr()['TB']
selected_features = corr_TB[(corr_TB.abs() > 0.5) & (corr_TB.index != 'TB')].index.tolist()
print("Selected features:", selected_features)

X = statcast[selected_features].dropna()
y = statcast.loc[X.index, 'TB']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


model = LinearRegression()
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
print(f"R² Score:  {r2_score(y_test, y_pred):.4f}")
print(f"RMSE:      {np.sqrt(mean_squared_error(y_test, y_pred)):.4f}")

coef_df = pd.DataFrame({
    'Feature': selected_features,
    'Coefficient': model.coef_
}).sort_values('Coefficient', key=abs, ascending=False)
print(coef_df)
print(f"Intercept: {model.intercept_:.4f}")

Selected features: ['Pitches', 'Zone_%', 'Barrel_%', 'Batted_Balls', 'Barrels', 'Hard_Hit_%']
R² Score:  -0.1276
RMSE:      22.8169
        Feature  Coefficient
2      Barrel_%    -2.072841
4       Barrels     1.897472
5    Hard_Hit_%     1.277822
1        Zone_%    -0.729468
3  Batted_Balls     0.250446
0       Pitches     0.010808
Intercept: 18.2355


In [12]:
def run_regression_analysis(target, features, data):
    print(f"\n{'='*60}")
    print(f"  TARGET VARIABLE: {target}")
    print(f"{'='*60}")

    cols = features + [target]
    corr = data[cols].corr()[target]
    selected_features = corr[(corr.abs() > 0.5) & (corr.index != target)].index.tolist()
    print("Selected features:", selected_features)

    results = {'target': target, 'selected_features': selected_features, 'models': {}}

    if len(selected_features) == 0:
        print(f"No features with |correlation| > 0.5 found for {target}. Skipping.")
        return results

    X = data[selected_features].dropna()
    y = data.loc[X.index, target]

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    X_scaled = scaler.fit_transform(X)

    # --- Linear Regression ---
    lr_model = LinearRegression()
    lr_model.fit(X_train_scaled, y_train)
    lr_pred = lr_model.predict(X_test_scaled)
    lr_cv = cross_val_score(lr_model, X_scaled, y, cv=5, scoring='r2')
    results['models']['Linear Regression'] = {
        'r2': r2_score(y_test, lr_pred),
        'rmse': np.sqrt(mean_squared_error(y_test, lr_pred)),
        'cv_r2_mean': lr_cv.mean(),
        'cv_r2_std': lr_cv.std()
    }

    # --- Ridge Regression ---
    ridge_model = RidgeCV(alphas=[0.1, 1.0, 10.0, 100.0], cv=5)
    ridge_model.fit(X_train_scaled, y_train)
    ridge_pred = ridge_model.predict(X_test_scaled)
    ridge_cv = cross_val_score(ridge_model, X_scaled, y, cv=5, scoring='r2')
    results['models']['Ridge Regression'] = {
        'r2': r2_score(y_test, ridge_pred),
        'rmse': np.sqrt(mean_squared_error(y_test, ridge_pred)),
        'cv_r2_mean': ridge_cv.mean(),
        'cv_r2_std': ridge_cv.std(),
        'best_alpha': ridge_model.alpha_
    }

    # --- Lasso Regression ---
    lasso_model = LassoCV(cv=5, random_state=42)
    lasso_model.fit(X_train_scaled, y_train)
    lasso_pred = lasso_model.predict(X_test_scaled)
    lasso_cv = cross_val_score(lasso_model, X_scaled, y, cv=5, scoring='r2')
    results['models']['Lasso Regression'] = {
        'r2': r2_score(y_test, lasso_pred),
        'rmse': np.sqrt(mean_squared_error(y_test, lasso_pred)),
        'cv_r2_mean': lasso_cv.mean(),
        'cv_r2_std': lasso_cv.std(),
        'best_alpha': lasso_model.alpha_,
        'coefficients': dict(zip(selected_features, lasso_model.coef_))
    }

    # --- Random Forest ---
    rf_model = RandomForestRegressor(n_estimators=100, random_state=42)
    rf_model.fit(X_train, y_train)
    rf_pred = rf_model.predict(X_test)
    rf_cv = cross_val_score(rf_model, X, y, cv=5, scoring='r2')
    results['models']['Random Forest'] = {
        'r2': r2_score(y_test, rf_pred),
        'rmse': np.sqrt(mean_squared_error(y_test, rf_pred)),
        'cv_r2_mean': rf_cv.mean(),
        'cv_r2_std': rf_cv.std(),
        'feature_importances': dict(zip(selected_features, rf_model.feature_importances_))
    }

    # Print summary
    print(f"\n===== MODEL COMPARISON: {target} =====")
    for name, m in results['models'].items():
        print(f"{name:<25} R²: {m['r2']:.4f}  CV R²: {m['cv_r2_mean']:.4f}  RMSE: {m['rmse']:.4f}")

    return results


In [13]:
def generate_pdf_report(all_results, filename="regression_report.pdf"):
    doc = SimpleDocTemplate(filename, pagesize=letter,
                            rightMargin=0.75*inch, leftMargin=0.75*inch,
                            topMargin=0.75*inch, bottomMargin=0.75*inch)
    styles = getSampleStyleSheet()
    story = []

    # Custom styles
    title_style = ParagraphStyle('CustomTitle', parent=styles['Title'], fontSize=20, spaceAfter=10)
    heading_style = ParagraphStyle('CustomHeading', parent=styles['Heading1'], fontSize=14,
                                   textColor=colors.HexColor('#2C3E50'), spaceAfter=6)
    subheading_style = ParagraphStyle('SubHeading', parent=styles['Heading2'], fontSize=11,
                                      textColor=colors.HexColor('#2980B9'), spaceAfter=4)
    normal_style = styles['Normal']
    header_style = ParagraphStyle('TableHeader', parent=styles['Normal'], textColor=colors.white, 
                                  fontName='Helvetica-Bold', fontSize=9, alignment=1)

    # Title Page
    story.append(Spacer(1, 0.8*inch))
    story.append(Paragraph("Baseball Statcast", title_style))
    story.append(Paragraph("Regression Analysis Report", title_style))
    story.append(Spacer(1, 0.3*inch))
    story.append(Paragraph("Targets: TB, BB, AB, H", styles['Heading2']))
    story.append(Paragraph("Models: Linear, Ridge, Lasso, Random Forest", styles['Heading2']))
    story.append(PageBreak())

    # Summary comparison table across all targets
    story.append(Paragraph("Overall Model Performance Summary", heading_style))
    story.append(Spacer(1, 0.1*inch))

    summary_data = [[
    Paragraph('Target', header_style),
    Paragraph('Best Model', header_style),
    Paragraph('R<super>2</super>', header_style),
    Paragraph('CV R<super>2</super>', header_style),
    Paragraph('RMSE', header_style)
]]
    for res in all_results:
        if not res['models']:
            continue
        best_model = max(res['models'].items(), key=lambda x: x[1]['r2'])
        name, m = best_model
        summary_data.append([
            res['target'], name,
            f"{m['r2']:.4f}",
            f"{m['cv_r2_mean']:.4f} ± {m['cv_r2_std']:.4f}",
            f"{m['rmse']:.4f}"
        ])

    summary_table = Table(summary_data, colWidths=[0.8*inch, 1.8*inch, 0.9*inch, 1.8*inch, 0.9*inch])
    summary_table.setStyle(TableStyle([
        ('BACKGROUND', (0, 0), (-1, 0), colors.HexColor('#2C3E50')),
        ('TEXTCOLOR', (0, 0), (-1, 0), colors.white),
        ('FONTNAME', (0, 0), (-1, 0), 'Helvetica-Bold'),
        ('FONTSIZE', (0, 0), (-1, -1), 8),
        ('ALIGN', (0, 0), (-1, -1), 'CENTER'),
        ('ROWBACKGROUNDS', (0, 1), (-1, -1), [colors.HexColor('#ECF0F1'), colors.white]),
        ('GRID', (0, 0), (-1, -1), 0.5, colors.grey),
        ('BOTTOMPADDING', (0, 0), (-1, -1), 6),
        ('TOPPADDING', (0, 0), (-1, -1), 6),
    ]))
    story.append(summary_table)
    story.append(PageBreak())

    # Detailed section per target
    for res in all_results:
        target = res['target']
        story.append(Paragraph(f"Target Variable: {target}", heading_style))

        # Selected features
        story.append(Paragraph("Selected Features (|correlation| > 0.5):", subheading_style))
        if res['selected_features']:
            story.append(Paragraph(", ".join(res['selected_features']), normal_style))
        else:
            story.append(Paragraph("No features met the threshold.", normal_style))
            story.append(PageBreak())
            continue

        story.append(Spacer(1, 0.15*inch))

        # Model comparison table
        story.append(Paragraph("Model Comparison", subheading_style))
        model_data = [[
            Paragraph('Model', header_style),
            Paragraph('R<super>2</super>', header_style),
            Paragraph('CV R<super>2</super>', header_style),
            Paragraph('RMSE', header_style),
            Paragraph('Best Alpha', header_style)
        ]]

        for model_name, m in res['models'].items():
            alpha = f"{m.get('best_alpha', 'N/A'):.4f}" if isinstance(m.get('best_alpha'), float) else 'N/A'
            model_data.append([
                model_name,
                f"{m['r2']:.4f}",
                f"{m['cv_r2_mean']:.4f} ± {m['cv_r2_std']:.4f}",
                f"{m['rmse']:.4f}",
                alpha
            ])

        model_table = Table(model_data, colWidths=[1.6*inch, 0.9*inch, 1.8*inch, 0.9*inch, 0.9*inch])
        model_table.setStyle(TableStyle([
            ('BACKGROUND', (0, 0), (-1, 0), colors.HexColor('#2980B9')),
            ('TEXTCOLOR', (0, 0), (-1, 0), colors.white),
            ('FONTNAME', (0, 0), (-1, 0), 'Helvetica-Bold'),
            ('FONTSIZE', (0, 0), (-1, -1), 8),
            ('ALIGN', (0, 0), (-1, -1), 'CENTER'),
            ('ROWBACKGROUNDS', (0, 1), (-1, -1), [colors.HexColor('#EBF5FB'), colors.white]),
            ('GRID', (0, 0), (-1, -1), 0.5, colors.grey),
            ('BOTTOMPADDING', (0, 0), (-1, -1), 3),
            ('TOPPADDING', (0, 0), (-1, -1), 3),
        ]))
        story.append(model_table)
        story.append(Spacer(1, 0.2*inch))

        # Lasso coefficients
        if 'Lasso Regression' in res['models'] and 'coefficients' in res['models']['Lasso Regression']:
            story.append(Paragraph("Lasso Coefficients", subheading_style))
            coef_data = [['Feature', 'Coefficient']]
            sorted_coefs = sorted(res['models']['Lasso Regression']['coefficients'].items(),
                                  key=lambda x: abs(x[1]), reverse=True)
            for feat, coef in sorted_coefs:
                coef_data.append([feat, f"{coef:.4f}"])

            coef_table = Table(coef_data, colWidths=[3*inch, 1.5*inch])
            coef_table.setStyle(TableStyle([
                ('BACKGROUND', (0, 0), (-1, 0), colors.HexColor('#2980B9')),
                ('TEXTCOLOR', (0, 0), (-1, 0), colors.white),
                ('FONTNAME', (0, 0), (-1, 0), 'Helvetica-Bold'),
                ('FONTSIZE', (0, 0), (-1, -1), 9),
                ('ALIGN', (1, 0), (1, -1), 'CENTER'),
                ('ROWBACKGROUNDS', (0, 1), (-1, -1), [colors.HexColor('#EBF5FB'), colors.white]),
                ('GRID', (0, 0), (-1, -1), 0.5, colors.grey),
                ('BOTTOMPADDING', (0, 0), (-1, -1), 3),
                ('TOPPADDING', (0, 0), (-1, -1), 3),
            ]))
            story.append(coef_table)
            story.append(Spacer(1, 0.2*inch))

        # Random Forest feature importances
        if 'Random Forest' in res['models'] and 'feature_importances' in res['models']['Random Forest']:
            story.append(Paragraph("Random Forest Feature Importances", subheading_style))
            imp_data = [['Feature', 'Importance']]
            sorted_imp = sorted(res['models']['Random Forest']['feature_importances'].items(),
                                key=lambda x: x[1], reverse=True)
            for feat, imp in sorted_imp:
                imp_data.append([feat, f"{imp:.4f}"])

            imp_table = Table(imp_data, colWidths=[3*inch, 1.5*inch])
            imp_table.setStyle(TableStyle([
                ('BACKGROUND', (0, 0), (-1, 0), colors.HexColor('#2980B9')),
                ('TEXTCOLOR', (0, 0), (-1, 0), colors.white),
                ('FONTNAME', (0, 0), (-1, 0), 'Helvetica-Bold'),
                ('FONTSIZE', (0, 0), (-1, -1), 9),
                ('ALIGN', (1, 0), (1, -1), 'CENTER'),
                ('ROWBACKGROUNDS', (0, 1), (-1, -1), [colors.HexColor('#EBF5FB'), colors.white]),
                ('GRID', (0, 0), (-1, -1), 0.5, colors.grey),
                ('BOTTOMPADDING', (0, 0), (-1, -1), 3),
                ('TOPPADDING', (0, 0), (-1, -1), 3),
            ]))
            story.append(imp_table)

        story.append(PageBreak())

    doc.build(story)
    print(f"\nPDF report saved to: {filename}")

In [ ]:
# --- Define base features ---
base_features = [
    'Pitches', 'Zone_%', 'Zone_Swing_%', 'Zone_Contact_%', 'Chase_%',
    'Chase_Contact_%', 'Edge_%', '1st_Pitch_Swing_%', 'Swing_%', 'Whiff_%',
    'Meatball_%', 'Meatball_Swing_%', 'GB_%', 'FB_%', 'LD_%', 'PU_%',
    'Pull_%', 'Straight_%', 'Oppo_%', 'Weak_%', 'Topped_%', 'Under_%',
    'Flare/Burner_%', 'Solid_%', 'Launch_Angle_Sweet-Spot_%', 'Barrel_%',
    'Batted_Balls', 'Barrels', 'Hard_Hit_%', 'Exit_Velocity', 'Launch_Angle'
]

# --- Run all targets and collect results ---
all_results = []
for target in ['TB', 'BB', 'AB', 'H']:
    result = run_regression_analysis(target, base_features, statcast)
    all_results.append(result)

# --- Generate PDF ---
generate_pdf_report(all_results, filename="single_team_statcast_regression_report.pdf")
os.startfile('single_team_statcast_regression_report.pdf')


  TARGET VARIABLE: TB
Selected features: ['Pitches', 'Zone_%', 'Barrel_%', 'Batted_Balls', 'Barrels', 'Hard_Hit_%']

===== MODEL COMPARISON: TB =====
Linear Regression         R²: -0.1276  CV R²: 0.8823  RMSE: 22.8169
Ridge Regression          R²: -1.8832  CV R²: 0.8806  RMSE: 36.4848
Lasso Regression          R²: 0.7609  CV R²: 0.9477  RMSE: 10.5060
Random Forest             R²: -6.4375  CV R²: 0.7190  RMSE: 58.5987

  TARGET VARIABLE: BB
Selected features: ['Pitches', 'Zone_%', 'Zone_Contact_%', 'Barrel_%', 'Batted_Balls', 'Barrels', 'Hard_Hit_%']

===== MODEL COMPARISON: BB =====
Linear Regression         R²: -406.8272  CV R²: -8.2423  RMSE: 115.4593
Ridge Regression          R²: -263.0699  CV R²: -6.3241  RMSE: 92.9074
Lasso Regression          R²: -364.0054  CV R²: -6.1473  RMSE: 109.2296
Random Forest             R²: -6.0110  CV R²: -0.7386  RMSE: 15.1384

  TARGET VARIABLE: AB
Selected features: ['Pitches', 'Zone_%', 'Chase_%', 'Barrel_%', 'Batted_Balls', 'Barrels', 'Hard_Hit_%